In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

In [0]:
%run /Workspace/Agmarknet/setup/utilities

In [0]:
print(gold_schema,silver_schema,bronze_schema)

In [0]:
dbutils.widgets.text('catalog','agmarknet')
dbutils.widgets.text('data source','Above_MSP_Prices')

In [0]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data source')
print(catalog,data_source)

In [0]:
file_path2025 = f's3://agmarknet-pc/raw/msp/2025/*.csv'
file_path2026 = f's3://agmarknet-pc/raw/msp/2026/*.csv'
print(file_path2025)
print(file_path2026)

###Read the file in text format

In [0]:
df1 = spark.read.text(file_path2025).select("*", "_metadata.file_name")
df2 = spark.read.text(file_path2026).select("*","_metadata.file_name")
df = df1.union(df2)

In [0]:
df.show()

In [0]:
df.select(F.col("file_name")).distinct().count()

In [0]:
###remove empty lines
df = df.filter(F.col("value").isNotNull() & (F.trim(F.col("value")) != ""))

In [0]:
df.show()

In [0]:
#add row number column
window = Window.partitionBy("file_name").orderBy(F.monotonically_increasing_id())
df = df.withColumn("row_num",F.row_number().over(window))

In [0]:
df.show()

In [0]:
df.count()

In [0]:
## Add row type column to identify Header, metadata, commodity,  group, data rows
df = df.withColumn("row_type", F.when(F.col("value").startswith("Group :"), "GROUP")
                   .when(F.col("value").startswith("State,"),"HEADER")
                   .when(F.col("value").startswith("Commodities reported Above (MSP) -"),"METADATA")
                   .when(F.col("value").rlike(r"^[A-Za-z][A-Za-z\s()&.-]*,"),"DATA")
                   .otherwise("COMMODITY")
)

In [0]:
##Display df 
df.filter(F.col("row_type").isin("METADATA")).distinct().count()

In [0]:
display(df)

###Extract the commodity group

In [0]:
df = (
    df.withColumn("Commodity_group",
                  F.when(F.col("value").startswith("Group :"),
                         F.regexp_extract("value",r"Group\s*:\s*(.*)",1)
                        ) 
                            )
)

In [0]:
display(df)

In [0]:
df.filter(F.col("Commodity_group").isNotNull()).show()

###Forward-fill the value of commodity group 

In [0]:
#Create the window per file
window_spec = (Window.partitionBy("file_name") 
    .orderBy("row_num").rowsBetween(Window.unboundedPreceding,Window.currentRow)
)

df = df.withColumn("Commodity_group",F.last("Commodity_group",ignorenulls=True).over(window_spec))
df.show()

In [0]:
#remove Meatadata rows 
df= df.filter(F.col("row_type").isin("DATA","HEADER","COMMODITY","GROUP"))
display(df)

In [0]:
df.select("value","row_num","Commodity_group","file_name","row_type").where(F.col("commodity_group")=="Cereals").show()

##Identify Commodity rows

In [0]:
df = df.withColumn(
    "Commodity",
    F.when(
        F.col("row_type")== "COMMODITY" ,
        F.regexp_replace(F.col("value"),'^"|"$', "")  ## Remove double quotes from commodity name   
    )
)

df.show()

In [0]:
##Forward-fill the value of commodity 
w = Window.partitionBy("file_name").orderBy("row_num").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df = df.withColumn("Commodity",F.last("Commodity", ignorenulls=True).over(w))
df.show()

In [0]:
df.select("Commodity").distinct().show(truncate=False)

In [0]:
#keep the data rows only
df = df.filter(F.col("row_type").isin("DATA"))
df.show()

In [0]:
df_M = df.select(F.col("value")).where(F.col("value").contains(","))
display(df_M)

In [0]:
# some of the Market center value contains comma, which shifts to the right column when saved as csv
#
df_csv = (
    df.withColumn("csv_line",
                  F.concat_ws(
                      ",",
                      F.concat(F.lit('"'),F.col("Commodity"),F.lit('"')),
                      F.concat(F.lit('"'),F.col("Commodity_group"),F.lit('"')),
                      F.col("value"),
                      F.concat(F.lit('"'),F.col("file_name"),F.lit('"'))
                  ))
)
display(df_csv)

In [0]:
df_csv.select(F.col("csv_line")).filter(F.col("csv_line").contains("Omchatinya Multi State Agro")).show(10,False)

In [0]:
# Writing csv_line as text to temp file
temp_path = "s3://agmarknet-pc/tmp1/"
df_text = df_csv.select(F.col("csv_line"))
#df_text.show(10)
df_text.write.mode("overwrite").text(temp_path)

In [0]:
#Read the file as csv
temp_path = "s3://agmarknet-pc/tmp1/"
final_df = spark.read.option("header","false") \
                     .option("quote",'"') \
                      .option("escape",'"') \
                      .csv(temp_path)

In [0]:
display(final_df)

In [0]:
#Rename the columns
final_df = (final_df.toDF(
    "Commodity",
    "Commodity_group",
    'State', 'Market_Center', 'Variety', 'Grade', 'Date', 'Arrivals', 'Unit_of_Arrivals', 'Modal_Price', 'Unit_of_Price',
    "file_name"
)
)

In [0]:
display(final_df)

In [0]:
ddf = final_df.filter(F.col("Market_Center").contains(","))
display(ddf)

In [0]:
print(final_df.columns)

In [0]:
final_df.count()

In [0]:
# split_col = F.split(F.col("value"), ",")

# df = df.select(
#     split_col.getItem(0).alias("State"),
#     split_col.getItem(1).alias("Market_Center"),
#     split_col.getItem(2).alias("Variety"),
#     split_col.getItem(3).alias("Grade"),
#     split_col.getItem(4).alias("Date"),
#     split_col.getItem(5).alias("Arrivals"),
#     split_col.getItem(6).alias("Unit_of_Arrivals"),
#     split_col.getItem(7).alias("Modal_Price"),
#     split_col.getItem(8).alias("Unit_of_Price"),
#                                "Commodity",
#                                "Commodity_group"
)

In [0]:
# df = df.withColumnRenamed("Market Center","Market_Center") \
#         .withColumnRenamed("Unit of Arrivals","Unit_of_Arrivals") \
#         .withColumnRenamed("Unit of Price","Unit_of_Price") \
#         .withColumnRenamed("Modal Price","Modal_Price")
        

In [0]:

# schema = """
#     State STRING, 
#     Market Center STRING,
#     Variety STRING,
#     Grade STRING,
#     Date STRING,
#     Arrivals DOUBLE,
#     Unit of Arrivals STRING,
#     Modal Price DOUBLE,
#     Unit of Price STRING
# """


In [0]:
# df= df.withColumn("parsed",F.from_csv(F.col("value"),
#                                       schema,
#                                       ))

In [0]:
# df= df.select("Commodity","Commodity_group","file_name","parsed.*")

###Save the data to bronze table

In [0]:
### Save the df  to bronze schema
final_df.write \
.format("delta") \
.option("overwriteSchema","true") \
.option("delta.enableChangeDataFeed", "true") \
.option("mergeSchema", "true") \
.mode("overwrite") \
.saveAsTable(f'{catalog}.{bronze_schema}.dim_{data_source}')

###silver layer transformation

In [0]:
silver_df = spark.sql(f'select * from {catalog}.{bronze_schema}.dim_{data_source}')
display(silver_df)
silver_df.count()

In [0]:
#extract Commodity name and MSP price from Commodity column
silver_df = (
    silver_df.withColumn(
        "Commodity_Name",
        F.regexp_extract(
            "Commodity",
            r"^(.*)\s+\(MSP:",
            1
        )
    )
    .withColumn(
        "MSP_Price",
        F.regexp_extract(
            "Commodity",
            r"MSP:\s*Rs\.\s*([\d.]+)",
            1
        ).cast("decimal(10,2)")
    )
)

In [0]:
display(silver_df)

In [0]:
silver_df.printSchema()

In [0]:
#cast Arrival quantity and Modal price to double
silver_df = silver_df.withColumn("Arrivals", F.col("Arrivals").cast("double")) \
            .withColumn("Modal_Price", F.col("Modal_Price").cast("double"))

In [0]:
# analyze date formats before transformation
df_silver_dates = spark.sql(f'select * from {catalog}.{bronze_schema}.dim_{data_source}')

df_formats = (
    df_silver_dates.withColumn(
        "date_format",
        F.when(F.col("Date").rlike(r"^\d{2}-\d{2}-\d{4}$"), "dd-MM-yyyy")
         .when(F.col("Date").rlike(r"^\d{2}/\d{2}/\d{4}$"), "dd/MM/yyyy")
         .when(F.col("Date").rlike(r"^\d{4}-\d{2}-\d{2}$"), "yyyy-MM-dd")
         .when(F.col("Date").rlike(r"^\d{4}/\d{2}/\d{2}$"), "yyyy/MM/dd")
         .when(F.col("Date").rlike(r"^\d{2}-[A-Za-z]{3}-\d{4}$"), "dd-MMM-yyyy")
         .otherwise("Unknown")
    )
)

df_formats.groupBy("date_format").count().show(truncate=False)

In [0]:
df_formats.filter(F.col("date_format")=="Unknown").show()

In [0]:
silver_df.printSchema()

In [0]:
silver_df = silver_df.withColumn("Date",F.to_date("Date","dd-MM-yyyy"))

In [0]:
silver_df.select("Date").show(10)

In [0]:
silver_df.printSchema()

In [0]:
silver_df = silver_df.withColumn("Arrivals",F.round(F.col("Arrivals"),2).cast("decimal(10,2)"))

###Save the table to silver schema

In [0]:
### Save the df  to silver schema
silver_df.write \
.format("delta") \
.option("overwriteSchema","true") \
.option("delta.enableChangeDataFeed", "true") \
.option("mergeSchema", "true") \
.mode("overwrite") \
.saveAsTable(f'{catalog}.{silver_schema}.dim_{data_source}')

###Gold layer

In [0]:
gold_df = spark.sql(f'select * from {catalog}.{silver_schema}.dim_{data_source}')
gold_df = gold_df.withColumn("Msp_Status",F.lit("Above MSP"))
display(gold_df)

In [0]:
gold_df.count()

In [0]:
#writing df to gold with MSP status
gold_df.write \
.format("delta") \
.option("overwriteSchema","true") \
.option("delta.enableChangeDataFeed", "true") \
.option("mergeSchema", "true") \
.mode("overwrite") \
.saveAsTable(f'{catalog}.{gold_schema}.dim_{data_source}')

###Create Fact MSP Table

In [0]:
comm_df = spark.sql(f'select * from {catalog}.{gold_schema}.dim_commodity')
comm_grp_df = spark.sql(f'select * from {catalog}.{gold_schema}.dim_commodity_group')
market_df = spark.sql(f'select * from {catalog}.{gold_schema}.dim_market')
ab_dim_df = spark.sql(f'select * from {catalog}.{gold_schema}.dim_above_msp_prices')

In [0]:
ab_dim_df.count()

In [0]:
# join comodity df and gold df to add commodity code
gold_df = ab_dim_df.alias("b").join(
   comm_df.alias("c"),
   (F.col("b.Commodity_Name") == F.col("c.Commodity")) &
   (F.col("b.Variety") == F.col("c.Variety")) &
   (F.col("b.Grade") == F.col("c.Grade")),
   "left"
).select(
   "c.Commodity_Code",
   "b.*"
)

In [0]:
gold_df.count()

In [0]:
display(gold_df)

In [0]:
# join comodity group df and gold df to add commodity group id
# 
gold_df1 = gold_df.alias("b").join(
   comm_grp_df.alias("c"),
   (F.col("b.Commodity_group") == F.col("c.Commodity_Group_Name")),
   "left"
).select(
   "c.Commodity_Group_Id",
   "b.*"
)
display(gold_df1)

In [0]:
gold_df1.count()

In [0]:
gold_df1.select("Commodity_Group_Id").distinct().count()

In [0]:
#writing df to gold with MSP status
gold_df1.write \
.format("delta") \
.option("overwriteSchema","true") \
.option("delta.enableChangeDataFeed", "true") \
.option("mergeSchema", "true") \
.mode("overwrite") \
.saveAsTable(f'{catalog}.{gold_schema}.temp_{data_source}')

In [0]:
gold_df1 = spark.sql(f'select * from {catalog}.{gold_schema}.temp_{data_source}')


In [0]:
gold_df1.count()

In [0]:
# join goldf1 and market df to get the market code

fact_abmsp_df = (
    gold_df1.alias("g")
    .join(
        market_df.alias("m"),
        (F.col("g.Market_Center") == F.col("m.market_name")) & (F.col("g.State") == F.col("m.state_name")),
        "left")
    .select("g.Commodity_Group_Id",
            "g.Commodity_Code",
            "g.MSP_Price",
            "m.market_id",
            "m.district_id",
            "m.state_id",
            "g.Date",
            "g.Arrivals",
            "g.Unit_of_Arrivals",
            "g.Modal_Price",
            "g.Unit_of_Price",
            "g.Msp_Status"
    )
)

In [0]:
fact_abmsp_df.count()

In [0]:
display(fact_abmsp_df)

In [0]:
#Save the df to table
fact_abmsp_df.write \
.format("delta") \
.option("overwriteSchema","true") \
.option("delta.enableChangeDataFeed", "true") \
.option("mergeSchema", "true") \
.mode("overwrite") \
.saveAsTable(f'{catalog}.{gold_schema}.fact_above_msp_prices')

In [0]:
spark.sql(f'drop table {catalog}.{gold_schema}.temp_{data_source}')